# Planning Simulations with Bayesian Optimization

You're a thermal engineer with a limited simulation budget. Each run of the simulator is expensive, so you have to choose which cold-plate designs to simulate, and in what order. This notebook compares an AI-guided strategy (Bayesian optimization) with random search under the same budget of 60 simulations.

It runs entirely in your browser (Pyodide). All physics and numbers are simplified and illustrative.

## 1. The simulator

One function, `evaluate(D_h, m_dot)`, returns a design's steady-state peak temperature and pressure drop. The two design variables are the channel hydraulic diameter `D_h` (mm) and the coolant flow rate `m_dot` (g/s). The goal is to minimize temperature while keeping the pressure drop within budget (`dP <= dP_budget`). Measurement noise is fixed per design, so evaluating the same design twice returns the same value, and the simulator stops after 60 calls.

In [ ]:
import hashlib, math
import numpy as np

def instance_seed(name):
    """Deterministic seed for a named problem instance."""
    return int(hashlib.sha256(str(name).encode()).hexdigest()[:8], 16)

def _frozen_noise(seed, vec):
    key = f'{seed}|' + '|'.join(f'{x:.6g}' for x in vec)
    d = hashlib.sha256(key.encode()).digest()
    u1 = (int.from_bytes(d[0:4],'big')+1)/(2**32+2)
    u2 = (int.from_bytes(d[4:8],'big')+1)/(2**32+2)
    return math.sqrt(-2*math.log(u1))*math.cos(2*math.pi*u2)

# reduced-order cold-plate physics (2 design variables: D_h in mm, m_dot in g/s)
_C = dict(rho=1050., mu=3.5e-3, k=0.37, Pr=25., cp=3400.); _KS=200.; _L=0.05; _Q=200.
BOUNDS = np.array([[0.5,5.0],[5.0,100.0]])  # D_h, m_dot

def _physics(D_h_mm, m_dot_gs, Q, N=16, AR=1.0, T_in=25.0, t_w_mm=1.5):
    D_h=D_h_mm*1e-3; m=m_dot_gs*1e-3; t_w=t_w_mm*1e-3
    w=D_h*(1+AR)/(2*AR); h=AR*w; A=w*h; perim=2*(w+h)
    v=(m/N)/(_C['rho']*A); Re=_C['rho']*v*D_h/_C['mu']
    gz=(D_h/_L)*max(Re,1.)*_C['Pr']
    nu_lam=3.66+(0.0668*gz)/(1+0.04*gz**(2/3)); nu_turb=0.023*max(Re,1.)**0.8*_C['Pr']**0.4
    if Re<2300: Nu=nu_lam
    elif Re>4000: Nu=nu_turb
    else:
        f=(Re-2300)/1700; Nu=(1-f)*nu_lam+f*nu_turb
    hc=Nu*_C['k']/D_h
    R=1/(hc*perim*_L*N)+t_w/(_KS*N*w*_L)
    T=T_in+Q*R+Q/(m*_C['cp'])
    if Re<2300: fr=64/max(Re,1.)
    elif Re>4000: fr=0.316*Re**-0.25
    else:
        f=(Re-2300)/1700; fr=(1-f)*(64/Re)+f*(0.316*Re**-0.25)
    dP=(fr*(_L/D_h)*(_C['rho']*v**2/2)+0.5*_C['rho']*v**2)/1000
    return T, dP

class BudgetExceeded(Exception): pass

class Oracle:
    def __init__(self, instance, budget=60, dP_budget=50., T_limit=36., noise_std=0.6):
        self.seed=instance_seed(instance); self.budget=budget
        self.dP_budget=dP_budget; self.T_limit=T_limit; self.noise_std=noise_std
        self.calls=0; self.reached_at=None
        self.Q=_Q*(1+0.10*(np.random.RandomState(self.seed%(2**31)).rand()-0.5))
    def evaluate(self, D_h, m_dot):
        if self.calls>=self.budget: raise BudgetExceeded(f'budget {self.budget} exhausted')
        self.calls+=1
        T,dP=_physics(D_h, m_dot, self.Q)
        T=T+self.noise_std*_frozen_noise(self.seed,[D_h,m_dot])
        feasible=bool(dP<=self.dP_budget)
        if self.reached_at is None and feasible and T<=self.T_limit: self.reached_at=self.calls
        return T, dP, feasible

print('oracle ready')

## 2. Bayesian optimization vs. random search

A small Gaussian-process surrogate (RBF kernel) with an expected-improvement acquisition function picks each next design. A full library such as Optuna would do the same job. Both campaigns run against the same simulator, bounds, and budget; only the planning strategy differs.

In [ ]:
def _k(A,B,l=0.25,sf=1.0):
    d2=np.sum(A**2,1)[:,None]+np.sum(B**2,1)[None,:]-2*A@B.T
    return sf**2*np.exp(-0.5*d2/l**2)
def _Phi(z): return 0.5*(1+np.vectorize(lambda t: math.erf(t/np.sqrt(2)))(z))
def _phi(z): return np.exp(-0.5*z**2)/np.sqrt(2*np.pi)

def run_bo(oracle, sampler_seed=0, n_init=8, n_cand=512):
    lo,hi=BOUNDS[:,0],BOUNDS[:,1]; rng=np.random.RandomState(sampler_seed)
    def unit(X): return (X-lo)/(hi-lo)
    X=rng.uniform(lo,hi,size=(n_init,2)); best=np.inf; traj=[]
    Xo=[]; yo=[]
    def step(x):
        nonlocal best
        T,dP,feas=oracle.evaluate(x[0],x[1]); pen=T if feas else T+50
        if feas: best=min(best,T)
        Xo.append(x); yo.append(pen)
        traj.append(dict(D_h=float(x[0]),m_dot=float(x[1]),best=(None if best==np.inf else best)))
    for i in range(n_init): step(X[i])
    while len(traj)<oracle.budget:
        Xa=np.array(Xo); ya=np.array(yo); ym=ya.mean()
        K=_k(unit(Xa),unit(Xa))+1e-3*np.eye(len(Xa)); L=np.linalg.cholesky(K)
        alpha=np.linalg.solve(L.T,np.linalg.solve(L,ya-ym))
        C=rng.uniform(lo,hi,size=(n_cand,2)); Ks=_k(unit(Xa),unit(C))
        mu=Ks.T@alpha+ym; v=np.linalg.solve(L,Ks); sd=np.sqrt(np.clip(1.0-np.sum(v**2,0),1e-9,None))
        imp=ya.min()-mu-0.01; z=imp/sd; ei=imp*_Phi(z)+sd*_phi(z)
        step(C[int(np.argmax(ei))])
    return traj

def run_random(oracle, sampler_seed=0):
    lo,hi=BOUNDS[:,0],BOUNDS[:,1]; rng=np.random.RandomState(sampler_seed+1); best=np.inf; traj=[]
    for _ in range(oracle.budget):
        x=rng.uniform(lo,hi); T,dP,feas=oracle.evaluate(x[0],x[1])
        if feas: best=min(best,T)
        traj.append(dict(D_h=float(x[0]),m_dot=float(x[1]),best=(None if best==np.inf else best)))
    return traj

INSTANCE='inst-14'; SAMPLER_SEED=7
ora=Oracle(INSTANCE); bo=run_bo(ora,SAMPLER_SEED); bo_reached=ora.reached_at
base=Oracle(INSTANCE); rnd=run_random(base,SAMPLER_SEED); rnd_reached=base.reached_at
print(f'AI-guided (BO):      reached target at sim {bo_reached}')
print(f'Uninformed (random): reached target at sim {rnd_reached}')
if bo_reached and rnd_reached: print(f'speedup: {rnd_reached/bo_reached:.1f}x fewer simulations')

In [ ]:
import matplotlib.pyplot as plt
def curve(tr):
    b=np.inf; out=[]
    for t in tr:
        v=t['best']; b=min(b, v if v is not None else np.inf); out.append(b if b<np.inf else np.nan)
    return out
fig,ax=plt.subplots(figsize=(7,4.2))
ax.plot(range(1,61),curve(bo),lw=2,label='AI-guided (Bayesian optimization)',color='#2d6a4f')
ax.plot(range(1,61),curve(rnd),lw=2,label='Uninformed (random search)',color='#adb5bd')
ax.axhline(36,ls='--',color='#b23a48',lw=1,label='engineering target (36 C)')
ax.set_xlabel('simulation (oracle evaluation)'); ax.set_ylabel('best feasible T_max (C)')
ax.legend(); ax.grid(alpha=.3); ax.set_title('AI-guided vs. uninformed simulation planning'); plt.show()

## Try it

- Change `INSTANCE` or `SAMPLER_SEED` and re-run. How consistent is the speedup?
- Shrink the budget. At what point does Bayesian optimization stop beating random search?
- Tighten `dP_budget`. How does the constraint change which designs the optimizer explores?